# Watch a Lock Queue Form

Companion notebook for the [Locks and Lock Queues](https://codemia.io/courses/postgresql_deep_dive/pg_locks_and_lock_queues) lesson on Codemia.

**What you will build**: a reproduction of the most common migration outage, on a live PostgreSQL server. One session holds an ordinary transaction open. A migration asks for `ACCESS EXCLUSIVE` on the same table and waits. Every query that arrives after it waits too, because lock requests queue in order. You will inspect the queue in `pg_locks` with `pg_blocking_pids`, measure the latency a steady stream of reads sees while it lasts, and then run the same migration with `lock_timeout` and retries so that reads are never blocked for more than a fraction of a second.

Runs on Colab with no GPU. Uses PostgreSQL, `psycopg2`, `numpy` and `matplotlib`.

## Setup

On Colab the next cell installs PostgreSQL with apt and starts it, which takes about a minute. Anywhere else, point the `PG_DSN` environment variable at a server you can create tables on, for example a Docker container started with `docker run -e POSTGRES_PASSWORD=postgres -p 5432:5432 postgres:14`.

In [ ]:
%pip install -q psycopg2-binary numpy matplotlib

In [ ]:
import os, sys, time, random, threading, subprocess
import psycopg2
import psycopg2.errors
import numpy as np
import matplotlib.pyplot as plt

if "google.colab" in sys.modules:
    def sh(cmd):
        subprocess.run(cmd, shell=True, check=True, stdout=subprocess.DEVNULL)
    sh("apt-get -qq update && apt-get -qq install -y postgresql postgresql-contrib")
    sh("service postgresql start")
    sh("sudo -u postgres psql -qc \"ALTER USER postgres PASSWORD 'postgres'\"")

DSN = os.environ.get("PG_DSN", "host=localhost user=postgres password=postgres dbname=postgres")

def connect():
    conn = psycopg2.connect(DSN)
    conn.autocommit = True
    return conn

db = connect()

def q(sql, args=None, conn=None):
    with (conn or db).cursor() as cur:
        cur.execute(sql, args)
        return cur.fetchall() if cur.description else None

print(q("SELECT version()")[0][0])

## 1. A table and some helpers

Each simulated client gets its own connection, because locks belong to sessions. `run_async` starts a statement on a background thread and returns immediately, which is how we get several sessions waiting at once from one notebook.

In [ ]:
q("DROP TABLE IF EXISTS accounts")
q("CREATE TABLE accounts (id int PRIMARY KEY, balance int)")
q("INSERT INTO accounts SELECT g, 100 FROM generate_series(1, 10000) g")

def session(name):
    conn = connect()
    q("SET application_name = %s", (name,), conn=conn)
    return conn

def run_async(conn, sql):
    result = {}
    def work():
        t = time.perf_counter()
        try:
            q(sql, conn=conn)
            result["status"] = "ok"
        except psycopg2.Error as e:
            result["status"] = type(e).__name__
        result["ms"] = (time.perf_counter() - t) * 1000
    th = threading.Thread(target=work, daemon=True)
    th.start()
    return th, result

def lock_view():
    return q("""
        SELECT a.application_name, l.mode, l.granted, pg_blocking_pids(a.pid) AS blocked_by, a.pid,
               left(a.query, 45)
        FROM pg_locks l JOIN pg_stat_activity a ON a.pid = l.pid
        WHERE l.relation = 'accounts'::regclass
        ORDER BY l.granted DESC, a.backend_start""")

def show_locks():
    names = {pid: app for app, _, _, _, pid, _ in lock_view()}
    print("%-12s %-20s %-8s %-14s %s" % ("session", "mode", "granted", "waits for", "query"))
    for app, mode, granted, blocked_by, pid, query in lock_view():
        waits = ",".join(names.get(p, str(p)) for p in blocked_by) or "-"
        print("%-12s %-20s %-8s %-14s %s" % (app, mode, granted, waits, query))

## 2. Build the queue by hand

Step one: a session begins a transaction and reads the table. It now holds `ACCESS SHARE` on `accounts` until it commits, even while it sits idle, which is what an application does when it opens a transaction and then waits on a slow external call.

In [ ]:
holder = session("holder")
holder.autocommit = False
q("SELECT count(*) FROM accounts", conn=holder)
show_locks()

Step two: a migration adds a column. `ALTER TABLE ... ADD COLUMN` with no default is instant once it runs, but it needs `ACCESS EXCLUSIVE`, which conflicts with every other mode including the holder's `ACCESS SHARE`. It waits.

In [ ]:
migration = session("migration")
mig_thread, mig_result = run_async(migration, "ALTER TABLE accounts ADD COLUMN note text")
time.sleep(0.5)
show_locks()

Step three: three ordinary reads arrive. `ACCESS SHARE` does not conflict with the holder's lock, so in principle they could run. They wait anyway, because a request that conflicts with one already waiting in the queue joins the queue behind it. `pg_blocking_pids` names the migration as their blocker, even though nothing they asked for conflicts with what the holder has.

In [ ]:
readers = [run_async(session(f"reader{i}"), "SELECT sum(balance) FROM accounts") for i in range(3)]
time.sleep(0.5)
show_locks()

One idle transaction plus one instant migration has stopped every read of the table. Commit the holder and the whole queue drains in order: the migration runs, then the reads.

In [ ]:
holder.commit()
mig_thread.join(); [th.join() for th, _ in readers]
print("migration:", mig_result)
for i, (_, r) in enumerate(readers):
    print(f"reader{i}:", r)

Each reader's latency is how long the holder stayed open after it arrived, not anything to do with its own work.

## 3. What steady traffic sees

Now replay the outage against a steady stream of reads, as an application would send. A background client issues a short query every 20 ms and records when it started and how long it took. The holder keeps its transaction open for 3 seconds, and the migration starts 0.5 seconds in.

In [ ]:
def traffic(duration_s, interval_s=0.02):
    conn = session("app")
    samples, t0 = [], time.perf_counter()
    while time.perf_counter() - t0 < duration_s:
        start = time.perf_counter()
        q("SELECT balance FROM accounts WHERE id = %s", (random.randint(1, 10000),), conn=conn)
        samples.append((start - t0, (time.perf_counter() - start) * 1000))
        time.sleep(max(0.0, interval_s - (time.perf_counter() - start)))
    conn.close()
    return np.array(samples)

def outage(migrate, hold_s=3.0, duration_s=5.0):
    out = {}
    app = threading.Thread(target=lambda: out.update(samples=traffic(duration_s)))
    app.start()
    time.sleep(0.3)
    h = session("holder"); h.autocommit = False
    q("SELECT count(*) FROM accounts", conn=h)
    time.sleep(0.5)
    m = threading.Thread(target=lambda: out.update(migration=migrate()))
    m.start()
    time.sleep(hold_s - 0.5)
    h.commit(); h.close()
    m.join(); app.join()
    return out

def naive_migration():
    conn = session("migration")
    t = time.perf_counter()
    q("ALTER TABLE accounts ADD COLUMN IF NOT EXISTS flag1 boolean", conn=conn)
    conn.close()
    return dict(attempts=1, ms=(time.perf_counter() - t) * 1000)

naive = outage(naive_migration)
s = naive["samples"]
print("migration:", naive["migration"])
print("reads: %d, p50 %.1f ms, max %.0f ms, reads slower than 100 ms: %d"
      % (len(s), np.percentile(s[:, 1], 50), s[:, 1].max(), (s[:, 1] > 100).sum()))

The read stream stops for the whole time the holder stays open after the migration arrives. Fewer reads complete in the window, because the client spends that time waiting on a single query.

## 4. The fix: lock_timeout and retry

Set `lock_timeout` for the migration session. If the lock is not granted within the timeout, the statement fails with `LockNotAvailable` and leaves the queue, so the reads behind it proceed. Retry after a pause until the lock is free.

In [ ]:
def safe_migration(timeout_ms=200, pause_s=0.3, max_attempts=50):
    conn = session("migration")
    q("SET lock_timeout = %s", (f"{timeout_ms}ms",), conn=conn)
    t = time.perf_counter()
    for attempt in range(1, max_attempts + 1):
        try:
            q("ALTER TABLE accounts ADD COLUMN IF NOT EXISTS flag2 boolean", conn=conn)
            break
        except psycopg2.errors.LockNotAvailable:
            time.sleep(pause_s)
    conn.close()
    return dict(attempts=attempt, ms=(time.perf_counter() - t) * 1000)

safe = outage(safe_migration)
s2 = safe["samples"]
print("migration:", safe["migration"])
print("reads: %d, p50 %.1f ms, max %.0f ms, reads slower than 100 ms: %d"
      % (len(s2), np.percentile(s2[:, 1], 50), s2[:, 1].max(), (s2[:, 1] > 100).sum()))

fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
for ax, (label, data) in zip(axes, [("no lock_timeout", s), ("lock_timeout 200 ms + retry", s2)]):
    ax.scatter(data[:, 0], data[:, 1], s=6)
    ax.axvspan(0.8, 3.3, color="tab:red", alpha=0.08, label="holder open, migration waiting")
    ax.set_ylabel("read latency, ms"); ax.set_title(label); ax.set_yscale("log")
axes[0].legend(loc="upper right"); axes[1].set_xlabel("seconds since start")
plt.tight_layout(); plt.show()

With the timeout, reads still stall, but never for longer than one `lock_timeout` at a time. The migration takes a few attempts and finishes as soon as the holder commits. In production the timeout is usually set between one and a few seconds, with a capped number of retries and an alert when they run out.

The other half of the fix is removing long holders in the first place: `idle_in_transaction_session_timeout` ends sessions that sit idle inside a transaction, and `statement_timeout` caps long queries.

In [ ]:
victim = session("forgetful"); victim.autocommit = False
q("SET idle_in_transaction_session_timeout = '1s'", conn=victim)
q("SELECT count(*) FROM accounts", conn=victim)
time.sleep(1.5)
try:
    q("SELECT 1", conn=victim)
except psycopg2.Error as e:
    print("session ended by the server:", type(e).__name__, str(e).strip().splitlines()[0])

## 5. Exercises

1. Replace `ADD COLUMN` with `CREATE INDEX` and then `CREATE INDEX CONCURRENTLY` in section 2. Which lock does each take, and do the readers queue?
2. In section 3, make the holder a writer (`UPDATE accounts SET balance = balance WHERE id = 1`) instead of a reader. Does a concurrent `SELECT` from the app still wait when there is no migration?
3. Sweep `timeout_ms` in `safe_migration` over 50, 200, 1000 and 3000 ms and plot the worst read latency and the number of attempts. Where would you set it for a table with a 100 ms latency budget?